In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 0 (the first): Enable synchronous CUDA error diagnosis
# ─────────────────────────────────────────────────────────────
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'   # Makes CUDA errors appear immediately at the place where they occur

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 1: Install and import libraries and set the seeds
# ─────────────────────────────────────────────────────────────
!pip install -q kagglehub transformers torch scikit-learn tensorboard tqdm

import random
import zipfile
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, RobertaConfig, RobertaForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
import kagglehub

# Set the seeds
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 2: Download and extract the data
# ─────────────────────────────────────────────────────────────
path = kagglehub.dataset_download("fizzbuzz/cleaned-toxic-comments")

if "archive.zip" in os.listdir(path):
    with zipfile.ZipFile(os.path.join(path, "archive.zip"), "r") as z:
        z.extractall(path)

train_csv = os.path.join(path, "train_preprocessed.csv")
df = pd.read_csv(train_csv)
df.rename(columns={"severe_tox": "severe_toxic"}, inplace=True)

label_cols = ["identity_hate", "insult", "obscene", "severe_toxic", "threat", "toxic"]
train_full = df[df.set == "train"].reset_index(drop=True)
train_df    = train_full.sample(frac=0.8, random_state=42)
val_df      = train_full.drop(train_df.index).reset_index(drop=True)


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 3: Define Dataset and DataLoader
# ─────────────────────────────────────────────────────────────
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len: int = 256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

MODEL_NAME  = "roberta-base"
BATCH_SIZE  = 256
NUM_WORKERS = max(1, os.cpu_count() - 1)

tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds    = ToxicDataset(train_df.comment_text.values, train_df[label_cols].values, tokenizer)
val_ds      = ToxicDataset(val_df.comment_text.values,   val_df[label_cols].values,   tokenizer)

train_loader= DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=True)
val_loader  = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 4: Set up the model from scratch
# ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = RobertaConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=512,
    num_hidden_layers=8,
    num_attention_heads=8,
    intermediate_size=2048,
    num_labels=len(label_cols),
    max_position_embeddings=512,
    problem_type="multi_label_classification",
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

model     = RobertaForSequenceClassification(config).to(device)
optimizer = AdamW(model.parameters(), lr=4e-5, weight_decay=1e-2)

EPOCHS      = 5
total_steps = len(train_loader) * EPOCHS
warmup_steps= int(0.1 * total_steps)
scheduler   = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

pos_weights = []
for c in label_cols:
    vals = train_df[c].astype(int).values
    neg, pos = (vals == 0).sum(), (vals == 1).sum()
    pos_weights.append(neg / (pos + 1e-10))
pos_weights = torch.tensor(pos_weights, dtype=torch.float).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)
scaler    = GradScaler()


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 5: TensorBoard and Early Stopping
# ─────────────────────────────────────────────────────────────
writer        = SummaryWriter("runs/toxic_custom_roberta")
best_val_loss = float("inf")
patience, counter, gstep = 3, 0, 0


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 6: Training loop with checking input\_ids values
# ─────────────────────────────────────────────────────────────
vocab_sz = model.config.vocab_size

for epoch in range(1, EPOCHS + 1):
    print(f"\n▶️ Epoch {epoch}/{EPOCHS}")
    model.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc="📦 Training", leave=False):
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        # 🔍 Debug: Make sure that all input\_ids are less than vocab\_size
        max_id = input_ids.max().item()
        if max_id >= vocab_sz:
            raise ValueError(f"Found input_id {max_id} ≥ vocab_size {vocab_sz}")

        with autocast():
            logits = model(input_ids, attention_mask=attention_mask).logits
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        gstep += 1
        if gstep % 100 == 0:
            writer.add_scalar("Train/Loss", loss.item(), gstep)
            writer.add_scalar("Train/LR", scheduler.get_last_lr()[0], gstep)

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(val_loader, desc="🔍 Validation", leave=False):
        with torch.no_grad(), autocast():
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            if input_ids.max().item() >= vocab_sz:
                raise ValueError(f"Found input_id ≥ vocab_size during validation")

            logits   = model(input_ids, attention_mask=attention_mask).logits
            val_loss+= criterion(logits, labels).item()

            probs    = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    all_preds    = np.vstack(all_preds)
    all_labels   = np.vstack(all_labels)
    roc_auc      = roc_auc_score(all_labels, all_preds, average="macro")
    preds_bin    = (all_preds > 0.5).astype(int)
    prec, recall, f1, _ = precision_recall_fscore_support(
        all_labels, preds_bin, average="macro", zero_division=0
    )

    writer.add_scalar("Val/Loss",     avg_val_loss, epoch)
    writer.add_scalar("Val/ROC_AUC",  roc_auc,       epoch)
    writer.add_scalar("Val/F1",       f1,            epoch)

    print(
        f"Train Loss={avg_train_loss:.4f} | "
        f"Val Loss={avg_val_loss:.4f} | "
        f"ROC-AUC={roc_auc:.4f} | F1={f1:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter       = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict()
        }, "best_model.pt")
        print("💾 The best model has been saved!")
    else:
        counter += 1
        if counter >= patience:
            print("🛑 Early stopping has been activated!")
            break


In [ ]:
# ─────────────────────────────────────────────────────────────
#Cell 7: Save the model and the tokenizer and quick testing
# ─────────────────────────────────────────────────────────────
model.save_pretrained("toxic_roberta_model")
tokenizer.save_pretrained("toxic_roberta_tokenizer")
writer.close()

# Quick test

checkpoint = torch.load("best_model.pt")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Illustrative input and individual predictions omitted from publication copy.


Historical cell 8 contained ten illustrative comments and individual predictions. It has been omitted from this publication copy. The unchanged original remains in the private recovery package. These examples were not a labeled test set.

In [ ]:
# --------------------------------------------------------------------
# Cell X: Collect metrics and generate plots for the final report
# --------------------------------------------------------------------
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

# 1) Create an output directory
os.makedirs("report_outputs", exist_ok=True)

# 2) Full classification report (precision / recall / F1 for each label)
report_dict = classification_report(
    all_labels,          # shape: (n_samples, 6)
    preds_bin,           # shape: (n_samples, 6)
    target_names=label_cols,
    output_dict=True,
    zero_division=0
)

# Save the report as JSON for later reference
with open("report_outputs/classification_report.json", "w") as f:
    json.dump(report_dict, f, indent=2)

print("\n=== Classification Report ===")
for lbl in label_cols:
    pr = report_dict[lbl]
    print(f"{lbl:13s} | P={pr['precision']:.3f}  R={pr['recall']:.3f}  F1={pr['f1-score']:.3f}")

macro = report_dict["macro avg"]
print(f"\nMacro-Avg      | P={macro['precision']:.3f}  R={macro['recall']:.3f}  F1={macro['f1-score']:.3f}")

# 3) Confusion matrix (one-vs-rest) for each label
for i, lbl in enumerate(label_cols):
    cm = confusion_matrix(all_labels[:, i], preds_bin[:, i])
    fig, ax = plt.subplots(figsize=(3, 3))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"Confusion Matrix – {lbl}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    for (j, k), v in np.ndenumerate(cm):
        ax.text(k, j, str(v), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(f"report_outputs/cm_{lbl}.png")
    plt.close(fig)

# 4) ROC curves for each label + macro-average curve
plt.figure(figsize=(6, 5))
macro_tpr = np.zeros(101)

for i, lbl in enumerate(label_cols):
    fpr, tpr, _ = roc_curve(all_labels[:, i], all_preds[:, i])
    roc_auc     = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{lbl} (AUC={roc_auc:.2f})")
    # Interpolate to build a macro curve
    interp_tpr = np.interp(np.linspace(0, 1, 101), fpr, tpr)
    macro_tpr += interp_tpr



# 5) Export the main metrics table as CSV for easy inclusion in the report
metrics_table = pd.DataFrame.from_dict(report_dict).T
metrics_table.to_csv("report_outputs/metrics_table.csv", float_format="%.3f")

print("\n✓ Metrics, confusion matrices, and ROC curves saved to 'report_outputs/'")
macro_tpr /= len(label_cols)
macro_auc = auc(np.linspace(0, 1, 101), macro_tpr)
plt.plot(np.linspace(0, 1, 101), macro_tpr, 'k--',
         label=f"Macro Avg (AUC={macro_auc:.2f})", lw=2)
plt.plot([0, 1], [0, 1], 'k:', lw=1)
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves – Toxic Comment Labels")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig("report_outputs/roc_curves.png")
plt.close()

In [ ]:
# Install graphviz if not present and generate a simple architecture diagram
import subprocess, os, textwrap, pathlib, json, sys, importlib, types, typing, math, builtins
# Install python graphviz package if needed
import sys, subprocess, json, os, textwrap, pathlib, typing, builtins, importlib, math, types
# Try to import graphviz; install if missing
try:
    import graphviz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "graphviz"])
    import graphviz
# Create the diagram
from graphviz import Digraph

dot = Digraph(comment="Eight-layer RoBERTa architecture")

dot.attr(rankdir='LR', fontsize='10')

# Nodes
dot.node('A', 'Input Text\n(comment)')
dot.node('B', 'Tokenizer\n(Byte-Pair Encoding)')
dot.node('C', 'Embedding Layer')
for i in range(1, 9):
    dot.node(f'Enc{i}', f'Encoder Block {i}')
dot.node('P', 'Pooling\n([CLS] token)')
dot.node('D', 'Dense Head\n(6 Sigmoid neurons)')
dot.node('O', 'Six Label\nProbabilities')

# Edges
dot.edges(['AB', 'BC'])
dot.edge('C', 'Enc1')
for i in range(1, 8):
    dot.edge(f'Enc{i}', f'Enc{i+1}')
dot.edge('Enc8', 'P')
dot.edge('P', 'D')
dot.edge('D', 'O')

# Render diagram
output_path = '/mnt/data/model_diagram'
dot.render(output_path, format='png', cleanup=True)

print(f"Diagram saved to {output_path}.png")
